# Colab 09 — ¿Puedo medir una masa desconocida sin usar una balanza?

**Laboratorio 1 · Departamento de Física · FCEN-UBA**

Clase 9 — 07/10

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/charlyacha/Labo1-colabs/blob/main/09_Senales_periodicas_y_determinacion_del_periodo.ipynb)

No es un truco: así se pesa a los astronautas en órbita, donde una balanza no mide nada. Un resorte, un cronómetro y la relación $T = 2\pi\sqrt{m/k}$ alcanzan.

**Al terminar vas a poder:** detectar máximos en una señal, determinar un período con la mejor precisión que permiten tus datos, y despejar una masa con su incerteza.

---

### Antes de tocar nada

Andá a **Archivo → Guardar una copia en Drive**. Vas a trabajar sobre tu copia:
lo que escribas acá sin copiar primero no se guarda en ningún lado.

Este cuaderno se recorre **de arriba hacia abajo**. Las celdas no son
independientes: cada una usa lo que definieron las anteriores. Si algo tira
`NameError`, casi siempre es porque salteaste una celda.

In [ ]:
import os

if not os.path.exists("lab1_utils.py"):
    !wget -q -O lab1_utils.py https://raw.githubusercontent.com/charlyacha/Labo1-colabs/main/lab1_utils.py

import numpy as np
import matplotlib.pyplot as plt
import lab1_utils as lab

lab.estilo_lab1()
print("Listo. numpy", np.__version__)

### 1. La señal

Un sensor de posición registra la oscilación. Lo primero, como siempre, es
mirarla.

In [ ]:
generador = np.random.default_rng(9)

T_real = 0.8123
t = np.arange(0, 12, 0.005)
x = 0.052*np.cos(2*np.pi*t/T_real + 0.3) + generador.normal(0, 0.0015, len(t))

fig, ax = plt.subplots()
ax.plot(t, x, "-", lw=0.8)
ax.set_xlabel("Tiempo (s)")
ax.set_ylabel("Posición (m)")
ax.set_xlim(0, 4)
plt.show()

### 2. Encontrar los máximos

`scipy.signal.find_peaks` busca máximos locales. Sin restricciones encuentra
también los que produce el ruido, así que hay que darle al menos una: la
altura mínima, o la distancia mínima entre picos.

In [ ]:
from scipy.signal import find_peaks

indices_ingenuo, _ = find_peaks(x)
indices, _ = find_peaks(x, height=0.02, distance=int(0.5*T_real/0.005))

print(f"sin restricciones: {len(indices_ingenuo)} picos (la mayoría es ruido)")
print(f"con restricciones: {len(indices)} picos")

t_picos = t[indices]

fig, ax = plt.subplots()
ax.plot(t, x, "-", lw=0.8, alpha=0.6)
ax.plot(t_picos, x[indices], "v", color="crimson", ms=8)
ax.set_xlabel("Tiempo (s)")
ax.set_ylabel("Posición (m)")
ax.set_xlim(0, 4)
plt.show()

### 3. Dos maneras de sacar el período, y una es mucho mejor

**Manera 1 (la intuitiva):** promediar las diferencias entre picos
consecutivos.

**Manera 2:** ajustar una recta a $t_n$ en función de $n$. La pendiente es
el período.

Parecen equivalentes. No lo son, y la razón es preciosa: la suma de las
diferencias consecutivas **telescopea**.

$$\frac{1}{N-1}\sum_{n=1}^{N-1}(t_{n+1} - t_n) = \frac{t_N - t_1}{N-1}$$

O sea que el promedio de diferencias usa **solamente el primer y el último
pico** y tira toda la información de los del medio. El ajuste los usa todos.

In [ ]:
diferencias = np.diff(t_picos)
T_prom = diferencias.mean()
sT_prom = np.std(diferencias, ddof=1)/np.sqrt(len(diferencias))

print(f"promedio de diferencias : {T_prom:.6f} s")
print(f"(t_ultimo - t_primero)/(N-1) = "
      f"{(t_picos[-1]-t_picos[0])/(len(t_picos)-1):.6f} s")
print("Son el mismo número, exactamente. Ahí está el telescopeo.")

In [ ]:
def recta(x, a, b):
    return a*x + b


n = np.arange(len(t_picos))
p, e, _ = lab.ajustar(recta, n, t_picos, nombres=["T (s)", "t0 (s)"])

T_ajuste, sT_ajuste = p[0], e[0]

print()
print(f"por promedio de diferencias : {lab.formatear(T_prom, sT_prom, 's')}")
print(f"por ajuste de t_n vs n      : {lab.formatear(T_ajuste, sT_ajuste, 's')}")
print(f"el ajuste es {sT_prom/sT_ajuste:.1f} veces más preciso, con los mismos datos")

In [ ]:
fig, axes = lab.grafico_con_residuos(
    n, t_picos, recta, p,
    xlabel="Número de oscilación, n", ylabel="Tiempo del pico (s)",
    etiqueta_modelo="ajuste lineal")
plt.show()

Si en ese panel de residuos ves una tendencia, el período **está cambiando**
a lo largo de la medición: hay amortiguamiento que corre la frecuencia, o el
sistema se está calentando, o perdiste un pico en el conteo. Perder un pico
es el error más común y se ve como un escalón.

### 4. Período contra masa

$$T = 2\pi\sqrt{\frac{m + m_{ef}}{k}}
\quad\Longrightarrow\quad
T^2 = \frac{4\pi^2}{k}\,m + \frac{4\pi^2}{k}m_{ef}$$

En la variable $T^2$ el modelo es una recta: la pendiente da $k$ y la
ordenada al origen da la **masa efectiva del resorte**, que para un resorte
homogéneo vale $m_{res}/3$ (hay literatura al respecto en el *American
Journal of Physics*; no es un ajuste ad hoc).

Ojo con la linealización: al pasar de $T$ a $T^2$ las barras de error se
transforman como $\sigma_{T^2} = 2T\,\sigma_T$. **No** son uniformes aunque
$\sigma_T$ lo sea, así que el ajuste tiene que ser ponderado.

In [ ]:
k_real, mef_real = 24.8, 0.0081

masas = np.array([0.100, 0.150, 0.200, 0.250, 0.300, 0.350, 0.400])
T_med = 2*np.pi*np.sqrt((masas + mef_real)/k_real)
T_med = T_med + generador.normal(0, 0.0008, size=len(masas))
sT = np.full(len(masas), 0.0008)

T2 = T_med**2
sT2 = 2*T_med*sT

print("  m (kg)    T (s)      T² (s²)    sigma_T2")
for m_i, T_i, T2_i, s_i in zip(masas, T_med, T2, sT2):
    print(f"  {m_i:.3f}   {T_i:.4f}    {T2_i:.5f}    {s_i:.5f}")

In [ ]:
p2, e2, _ = lab.ajustar(recta, masas, T2, yerr=sT2,
                        nombres=["pendiente (s²/kg)", "ordenada (s²)"])

k_dinamico = 4*np.pi**2/p2[0]
sk_dinamico = 4*np.pi**2*e2[0]/p2[0]**2

m_efectiva = p2[1]/p2[0]
sm_efectiva = m_efectiva*np.sqrt((e2[1]/p2[1])**2 + (e2[0]/p2[0])**2)

print()
lab.reportar(k_dinamico, sk_dinamico, "N/m", nombre="k dinámico")
lab.reportar(m_efectiva*1000, sm_efectiva*1000, "g", nombre="masa efectiva")

In [ ]:
fig, axes = lab.grafico_con_residuos(
    masas, T2, recta, p2, yerr=sT2, normalizar_residuos=True,
    xlabel="Masa colgada (kg)", ylabel="T² (s²)")
plt.show()

### 5. ¿Es el mismo k que el de la Clase 5?

Ésta es la verificación que hace valer todo el diseño del cuatrimestre: el
$k$ **estático** (elongación bajo carga) y el $k$ **dinámico** (período de
oscilación) son dos determinaciones independientes de la misma magnitud
física, obtenidas de experimentos que no tienen nada que ver entre sí.

In [ ]:
k_estatico, sk_estatico = 24.79, 0.12

lab.compatibilidad(k_estatico, sk_estatico, k_dinamico, sk_dinamico,
                   etiquetas=("k estático (Clase 5)", "k dinámico (Clase 9)"))

Si no dan compatibles, la explicación más habitual no es que la física esté
mal: es que uno de los dos tiene un sistemático. Candidatos: en el estático,
el resorte no arrancaba relajado; en el dinámico, la amplitud era tan grande
que el resorte se salió del régimen lineal, o te olvidaste la masa efectiva.

### 6. Pesar sin balanza

Con $k$ conocido, una masa desconocida sale de su período:

$$m = \frac{k T^2}{4\pi^2} - m_{ef}$$

In [ ]:
T_incognita, sT_incognita = 0.9142, 0.0009

m_incognita = k_dinamico*T_incognita**2/(4*np.pi**2) - m_efectiva

# Propagación: los tres términos, en cuadratura.
d_dk = T_incognita**2/(4*np.pi**2)
d_dT = 2*k_dinamico*T_incognita/(4*np.pi**2)
d_dmef = -1.0

sm = np.sqrt((d_dk*sk_dinamico)**2 + (d_dT*sT_incognita)**2
             + (d_dmef*sm_efectiva)**2)

lab.reportar(m_incognita*1000, sm*1000, "g", nombre="masa incógnita")

contrib = np.array([(d_dk*sk_dinamico)**2, (d_dT*sT_incognita)**2,
                    (d_dmef*sm_efectiva)**2])
print()
print("contribuciones a la varianza:")
for nombre, valor in zip(["k", "T", "masa efectiva"], 100*contrib/contrib.sum()):
    print(f"  {nombre:<15} {valor:5.1f} %")

La tabla de contribuciones dice qué mejorar si querés más precisión, y suele
sorprender: el período se mide fácil y muy bien, así que el error casi
siempre está dominado por $k$. Medir la incógnita más veces no sirve; medir
mejor el resorte, sí.

### 7. Ejercicios

1. Determiná el período de tu oscilación por los dos métodos y verificá el
   telescopeo con tus propios datos.
2. Pesá un objeto desconocido y contrastalo con la balanza. ¿A cuántos sigma
   estás?
3. Sacá la masa efectiva del ajuste y compará con $m_{res}/3$ pesando el
   resorte. ¿Se verifica el factor 1/3?
4. Repetí el análisis con solo los primeros 5 picos. ¿Cuánto empeora
   $\sigma_T$? Compará con la predicción del ajuste: la incerteza de la
   pendiente escala como $N^{-3/2}$, no como $N^{-1/2}$. ¿Por qué medir en
   una ventana más larga rinde tanto?

In [ ]:
# Espacio de trabajo para los ejercicios.